In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import SMOTE

df = pd.read_csv("water_potability.csv")
df.head()

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


In [ ]:
df.tail()

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
3271,4.668102,193.681735,47580.991603,7.166639,359.948574,526.424171,13.894419,66.687695,4.435821,1
3272,7.808856,193.553212,17329.802160,8.061362,NaN,392.449580,19.903225,NaN,2.798243,1
3273,9.419510,175.762646,33155.578218,7.350233,NaN,432.044783,11.039070,69.845400,3.298875,1
3274,5.126763,230.603758,11983.869376,6.303357,NaN,402.883113,11.168946,77.488213,4.708658,1
3275,7.874671,195.102299,17404.177061,7.509306,NaN,327.459760,16.140368,78.698446,2.309149,1


In [ ]:
df.columns

Index(['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity',
       'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability'],
      dtype='object')

In [ ]:
df.isnull().sum()

,0
ph,491
Hardness,0
Solids,0
Chloramines,0
Sulfate,781
Conductivity,0
Organic_carbon,0
Trihalomethanes,162
Turbidity,0
Potability,0


In [ ]:
def fill_with_median(x):
    return x.fillna(x.median())

In [ ]:
for col in ['ph', 'Sulfate', 'Trihalomethanes']:

    df[col] = df.groupby('Potability')[col].transform(
        fill_with_median
    )

In [ ]:
df.isnull().sum()

,0
ph,0
Hardness,0
Solids,0
Chloramines,0
Sulfate,0
Conductivity,0
Organic_carbon,0
Trihalomethanes,0
Turbidity,0
Potability,0


In [ ]:
X = df.drop('Potability', axis=1)
y = df['Potability']

In [ ]:
print(y.value_counts())

Potability
0    1998
1    1278
Name: count, dtype: int64


In [ ]:

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

In [ ]:
print(y_resampled.value_counts())

Potability
0    1998
1    1998
Name: count, dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=15, n_estimators=200, random_state=42)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=['High Risk (0)', 'Low Risk (1)']
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8

Classification Report:
               precision    recall  f1-score   support

High Risk (0)       0.81      0.78      0.80       400
 Low Risk (1)       0.79      0.82      0.80       400

     accuracy                           0.80       800
    macro avg       0.80      0.80      0.80       800
 weighted avg       0.80      0.80      0.80       800


Confusion Matrix:
[[312  88]
 [ 72 328]]


In [ ]:
risk_mapping = {
    0: "High Risk",
    1: "Low Risk"
}

In [ ]:
def predict_risk(new_data):
    prediction = model.predict(new_data)[0]

    risk = risk_mapping[prediction]

    reasons = []

    if risk == "High Risk":

        reasons.append(
            "The water quality pattern resembles samples associated with higher health risk."
        )

        reasons.append(
            "Poor water quality may increase the possibility of water-borne diseases."
        )

        reasons.append(
            "Water treatment is recommended before consumption."
        )

    else:

        reasons.append(
            "The water quality pattern resembles safer water samples in the dataset."
        )

        reasons.append(
            "The possibility of water-borne disease risk is relatively lower."
        )

        reasons.append(
            "Water quality indicators appear to be within safer ranges."
        )

    return risk, reasons

In [ ]:
sample_unknown = pd.DataFrame({
    "ph":[7.8],
    "Hardness":[190],
    "Solids":[18000],
    "Chloramines":[6.5],
    "Sulfate":[320],
    "Conductivity":[420],
    "Organic_carbon":[12],
    "Trihalomethanes":[65],
    "Turbidity":[3.8]
})

In [ ]:
risk, reasons = predict_risk(sample_unknown)

print("Risk Level:", risk)


for r in reasons:
    print("-", r)

Risk Level: Low Risk
- The water quality pattern resembles safer water samples in the dataset.
- The possibility of water-borne disease risk is relatively lower.
- Water quality indicators appear to be within safer ranges.


In [ ]:
sample_unknown2 = pd.DataFrame({
    "ph":[5.8],
    "Hardness":[260],
    "Solids":[35000],
    "Chloramines":[9.0],
    "Sulfate":[430],
    "Conductivity":[620],
    "Organic_carbon":[18],
    "Trihalomethanes":[105],
    "Turbidity":[6.5]
})
risk, reasons = predict_risk(sample_unknown2)

print("Risk Level:", risk)


for r in reasons:
    print("-", r)

Risk Level: High Risk
- The water quality pattern resembles samples associated with higher health risk.
- Poor water quality may increase the possibility of water-borne diseases.
- Water treatment is recommended before consumption.


In [ ]:
import joblib

joblib.dump(model, "water_risk_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [ ]:
features = [
    "ph",
    "Hardness",
    "Solids",
    "Chloramines",
    "Sulfate",
    "Conductivity",
    "Organic_carbon",
    "Trihalomethanes",
    "Turbidity"
]

joblib.dump(features, "features.pkl")

print("Features saved!")

Features saved!


In [ ]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 79.0 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib

model = joblib.load("water_risk_model.pkl")
features = joblib.load("features.pkl")


st.set_page_config(
    page_title="Water Risk Prediction",
    page_icon="💧",
    layout="wide"
)


st.title("💧 Smart Community Health Monitoring")
st.subheader("Water Quality Risk Prediction System")

st.write(
    "Enter the water-quality parameters below to predict "
    "the water risk level using a Random Forest Machine Learning model."
)


st.sidebar.header("Water Quality Parameters")

ph = st.sidebar.number_input(
    "pH",
    min_value=0.0,
    max_value=14.0,
    value=7.0
)

hardness = st.sidebar.number_input(
    "Hardness",
    min_value=0.0,
    value=200.0
)

solids = st.sidebar.number_input(
    "Solids",
    min_value=0.0,
    value=20000.0
)

chloramines = st.sidebar.number_input(
    "Chloramines",
    min_value=0.0,
    value=7.0
)

sulfate = st.sidebar.number_input(
    "Sulfate",
    min_value=0.0,
    value=300.0
)

conductivity = st.sidebar.number_input(
    "Conductivity",
    min_value=0.0,
    value=400.0
)

organic_carbon = st.sidebar.number_input(
    "Organic Carbon",
    min_value=0.0,
    value=15.0
)

trihalomethanes = st.sidebar.number_input(
    "Trihalomethanes",
    min_value=0.0,
    value=60.0
)

turbidity = st.sidebar.number_input(
    "Turbidity",
    min_value=0.0,
    value=4.0
)



input_data = pd.DataFrame({
    "ph": [ph],
    "Hardness": [hardness],
    "Solids": [solids],
    "Chloramines": [chloramines],
    "Sulfate": [sulfate],
    "Conductivity": [conductivity],
    "Organic_carbon": [organic_carbon],
    "Trihalomethanes": [trihalomethanes],
    "Turbidity": [turbidity]
})


st.header("📊 Entered Water Quality Data")

st.dataframe(
    input_data,
    use_container_width=True
)


if st.button("🔍 Predict Water Risk"):

    prediction = model.predict(input_data)[0]

    probability = model.predict_proba(input_data)[0]

    high_risk_probability = probability[0] * 100
    low_risk_probability = probability[1] * 100



    st.header("Prediction Result")


    if prediction == 0:

        st.error("🚨 HIGH RISK")

        st.write(
            "The water-quality pattern resembles samples "
            "associated with higher health risk."
        )

        st.write(
            "Water treatment is recommended before consumption."
        )

    else:

        st.success("✅ LOW RISK")

        st.write(
            "The water-quality pattern resembles safer "
            "water samples in the dataset."
        )

        st.write(
            "The possibility of water-borne disease risk "
            "is relatively lower."
        )



    st.subheader("Prediction Probability")

    col1, col2 = st.columns(2)

    with col1:
        st.metric(
            "High Risk Probability",
            f"{high_risk_probability:.2f}%"
        )

    with col2:
        st.metric(
            "Low Risk Probability",
            f"{low_risk_probability:.2f}%"
        )


    st.subheader("Risk Probability")

    chart_data = pd.DataFrame({
        "Risk Level": [
            "High Risk",
            "Low Risk"
        ],
        "Probability": [
            high_risk_probability,
            low_risk_probability
        ]
    })

    st.bar_chart(
        chart_data.set_index("Risk Level")
    )



st.sidebar.markdown("---")

st.sidebar.info(
    "Model: Random Forest Classifier\n\n"
    "Target: Potability\n\n"
    "0 = High Risk\n"
    "1 = Low Risk"
)

Writing app.py


In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!pip install pyngrok

In [ ]:
!ngrok config add-authtoken "3JUZmsiFf4stDtJPFKSUzCYteo3_65kq6jW8X1TKQYuxboyvU"

In [ ]:
!streamlit run app.py --server.port 8501 &>/content/streamlit.log &

In [ ]:
!cat /content/streamlit.log

In [ ]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8501)

print("Your Streamlit Dashboard:")
print(public_url)